# HW2 Overview

In this assignment, we will study language model. You will get the basic ideas of maximum likelihood estimation, smoothing, generate text documents from language models, and language model evaluation. 

We will reuse the same Yelp dataset and refer to each individual user review as a **document** (e.g., as in computing document frequency). You should reuse your JSON parser in this assignment.

The same pre-processing steps you have developed in HW1 will be used in this assignment, i.e., tokenization, stemming and normalization. Note: **NO** stopword removal is needed in this assignment. 



# Statistical Language Models

### 1. Maximum likelihood estimation for statistical language models with proper smoothing (50pts)

Use all the review documents to estimate a unigram language model $p(w)$ and two bigram language models (with different smoothing methods specified below). Note those language models are corpus-level models, i.e., aggregating all the words across different documents.

When estimating the bigram language models, using linear interpolation smoothing and absolute discount smoothing based on the unigram language model $p\_u(w)$ to get two different bigram language models accordingly, i.e., $p^L(w\_i|w\_{i-1})$ and $p^A(w\_i|w\_{i-1})$. In linear interpolation smoothing, set the parameter $\lambda=0.9$; and in absolute discount smoothing, set the parameter $\delta=0.1$.

Specifically, when estimating $p^L(w\_i|w\_{i-1})$ and $p^A(w\_i|w\_{i-1})$, you should use the unigram language model $p(w_i)$ as the reference language model in smoothing. For example, in linear interpolation smoothing, the resulting smoothing formula looks like this,

$$p^L(w\_i|w\_{i-1})=(1-\lambda) \frac{c(w\_{i-1}w\_i)}{c(w\_{i-1})} + \lambda p(w\_i)$$ 
where $c(w\_{i-1}w\_i)$ is the frequency of bigram $w\_{i-1}w_i$ in the whole corpus.

From the resulting two bigram language models, find the top 10 words that are most likely to follow the word "good", i.e., rank the words in a descending order by $p^L(w|good")$ and $p^A(w|good")$ and output the top 10 words. Are those top 10 words the same from these two bigram language models? Explain your observation.

*HINT: to reduce space complexity, you do not need to actually maintain a $V\times V$ array to store the counts and probabilities for the bigram language models. You can use a sparse data structure, e.g., hash map, to store the seen words/bigrams, and perform the smoothing on the fly, i.e., evoke some function calls to return the value of $p^L(w|good")$ and $p^A(w|good")$.* 

**What to submit**:

1. Paste your implementation of the linear interpolation smoothing and absolute discount smoothing.
2. The top 10 words selected from the corresponding two bigram language models.
3. Your explanation of the observations about the top words under those two bigram language models.



In [1]:
import json
import nltk
import re
import string
import os

yelp_restaurants = []
stop_words = set(nltk.corpus.stopwords.words('english'))
TTF = {}
bigram_count = {}
total_num_words = 0

for yelp_fp in os.listdir('./yelp/'):
    with open('./yelp/' + yelp_fp, "r") as restaurant:
        yelp_restaurants.append(json.load(restaurant))


for restaurant in yelp_restaurants:
    for review in restaurant['Reviews']:
        freq = {}
        # Normalize before tokenization to prevent empty tokens e.g."$ 1.1" -> ['', '11']
        tokens = re.sub(r"\b\d+\b", "#", review['Content']
            .translate(str.maketrans('', '', string.punctuation))).lower()

        # Tokenize by word
        tokens = nltk.word_tokenize(tokens)

        # # Stopword removal
        # tokens = [word for word in tokens if not word in stop_words]

        # Token stemming
        tokens = [nltk.stem.PorterStemmer().stem(token) for token in tokens]

        # Write back to the object
        # review['NormalizedContent'] = tokens

        # Sum the number of words on each doc
        total_num_words += len(tokens)

        # Calc frequency of words in a doc
        for token in tokens:
            freq[token] = freq.get(token, 0) + 1

        # Calc frequency of word given next word in doc
        for i in range(len(tokens) - 1):
            current_word = tokens[i]
            next_word = tokens[i+1]

            if bigram_count.get(current_word, None) is None:
                bigram_count[current_word] = {}
            bigram_count[current_word][next_word] = bigram_count[current_word].get(next_word, 0) + 1

        for token in freq.keys():
            TTF[token] = TTF.get(token, 0) + freq[token]


# sort TTF
TTF = {key: val 
       for key, val in 
       sorted(TTF.items(), key=lambda pair:pair[1], reverse=True)}
# print('TTF: ', TTF)

unigram_prob = {key: val / total_num_words
       for key, val in TTF.items()}
# print(unigram_prob)

In [2]:
def li_smoothing(prev_word, w, prev_w_count, TTF_sum):
    return (1-li_lambda)*bigram_count[prev_word].get(w, 0)/prev_w_count + li_lambda*(TTF[w]/TTF_sum)

def ad_smoothing(prev_w, w, num_uniq_bigrams_given_word, total_bigram_count_given_word):
    return (max(bigram_count[prev_w].get(w, 0) - ad_delta, 0) + ad_delta * num_uniq_bigrams_given_word * unigram_prob[w]) / total_bigram_count_given_word

def bigram_smoothing(prev_word):
    prev_w_count = TTF[prev_word]
    TTF_sum = sum(TTF.values())
    num_uniq_bigrams_given_word = len(bigram_count[prev_word])
    total_bigram_count_given_word = sum(bigram_count[prev_word].values())
    
    for word in unigram_prob:
        if li_smooted_bigram.get(prev_word, None) is None:
            li_smooted_bigram[prev_word] = {}
            ad_smooted_bigram[prev_word] = {}
        li_smooted_bigram[prev_word][word] = li_smoothing(prev_word, word, prev_w_count, TTF_sum)
        ad_smooted_bigram[prev_word][word] = ad_smoothing(prev_word, word, num_uniq_bigrams_given_word, total_bigram_count_given_word)

def sort_bigram(prev_word):
    li_smooted_bigram[prev_word] = {key: val 
        for key, val in 
        sorted(li_smooted_bigram[prev_word].items(), key=lambda pair:pair[1], reverse=True)}
    ad_smooted_bigram[prev_word] = {key: val 
        for key, val in 
        sorted(ad_smooted_bigram[prev_word].items(), key=lambda pair:pair[1], reverse=True)}


li_lambda = 0.9
ad_delta = 0.1
li_smooted_bigram = {}
ad_smooted_bigram = {}


print("number of unique words", len(TTF))

# Perform smoothing on bigram
print("smoothing bigrams...")
bigram_smoothing('good')

print("sanity check li_smooted_bigram: ", sum(li_smooted_bigram['good'].values()))
print("sanity check ad_smooted_bigram: ", sum(ad_smooted_bigram['good'].values()))

sort_bigram('good')

number of unique words 168177
smoothing bigrams...
sanity check li_smooted_bigram:  0.9979340875585738
sanity check ad_smooted_bigram:  1.0000000000034648


In [3]:
print("sanity check: ", sum(li_smooted_bigram['good'].values()))
print("sanity check: ", sum(ad_smooted_bigram['good'].values()))
print("linear interpolation smoothing: ", list(li_smooted_bigram['good'].keys())[:10])
print("absolute discount smoothing: ", list(ad_smooted_bigram['good'].keys())[:10])

sanity check:  0.9979340875585738
sanity check:  1.0000000000034646
linear interpolation smoothing:  ['the', 'and', 'i', 'a', 'to', 'but', 'it', 'wa', 'of', 'for']
absolute discount smoothing:  ['but', 'and', 'the', 'i', 'as', 'food', 'it', 'thing', 'for', 'too']


### 2. Generate text documents from a language model (40pts)

Fixing the document length to 20, generate 10 documents by sampling words from $p(w)$, $p^L(w\_i|w\_{i-1})$ and $p^A(w\_i|w\_{i-1})$ respectively.

*HINT: you can use $p(w)$ to generate the first word of a document and then sampling from the corresponding bigram language model when generating from $p^L(w\_i|w\_{i-1})$ and $p^A(w\_i|w\_{i-1})$.* 

**What to submit**:

1. Paste your implementation of the sampling procedure from a language model.
2. The 10 documents generated from $p(w)$, $p^L(w\_i|w\_{i-1})$ and $p^A(w\_i|w\_{i-1})$ accordingly, and the corresponding likelihood given by the used language model.

In [60]:
import numpy as np

document_length = 20
number_of_documents = 10

unigram_documents = [[word] for word in np.random.choice(list(unigram_prob.keys()), 10, p=list(unigram_prob.values()))]
li_documents = [[word] for word in np.random.choice(list(unigram_prob.keys()), 10, p=list(unigram_prob.values()))]
ad_documents = [[word] for word in np.random.choice(list(unigram_prob.keys()), 10, p=list(unigram_prob.values()))]

print("unigram_documents: ", unigram_documents)
print("li_documents: ", li_documents)
print("ad_documents: ", ad_documents)

for _ in range(document_length - 1):
    for i in range(number_of_documents):
        unigram_documents[i].append(np.random.choice(list(unigram_prob.keys()), 1, p=list(unigram_prob.values()))[0])

        # Generate word for Linear Interpolation smoothing
        prev_word = li_documents[i][-1]

        # Smoothing all bigrams given prev_word
        # print(f"smoothing bigrams for {prev_word}")
        prev_w_count = TTF[prev_word]
        TTF_sum = sum(TTF.values())
        for word in unigram_prob:
            if li_smooted_bigram.get(prev_word, None) is None:
                li_smooted_bigram[prev_word] = {}
            li_smooted_bigram[prev_word][word] = li_smoothing(prev_word, word, prev_w_count, TTF_sum)
        # normalize prob
        li_smoothed_sum = sum(li_smooted_bigram[prev_word].values())
        # print(f"sanity check({prev_word})(li non-normalized): ", li_smoothed_sum)
        for word in li_smooted_bigram[prev_word]:
            li_smooted_bigram[prev_word][word] /= li_smoothed_sum
        li_documents[i].append(np.random.choice(list(li_smooted_bigram[prev_word].keys()), 1, p=list(li_smooted_bigram[prev_word].values()))[0])


        # Generate word for Absolute Discount smoothing
        prev_word = ad_documents[i][-1]

        # Smoothing all bigrams given prev_word
        # print(f"smoothing bigrams for {prev_word}")
        num_uniq_bigrams_given_word = len(bigram_count[prev_word])
        total_bigram_count_given_word = sum(bigram_count[prev_word].values())
        for word in unigram_prob:
            if ad_smooted_bigram.get(prev_word, None) is None:
                ad_smooted_bigram[prev_word] = {}
            ad_smooted_bigram[prev_word][word] = ad_smoothing(prev_word, word, num_uniq_bigrams_given_word, total_bigram_count_given_word)
        # normalize prob
        ad_smoothed_sum = sum(ad_smooted_bigram[prev_word].values())
        # print(f"sanity check({prev_word})(ad non-normalized): ", ad_smoothed_sum)
        for word in ad_smooted_bigram[prev_word]:
            ad_smooted_bigram[prev_word][word] /= ad_smoothed_sum
        ad_documents[i].append(np.random.choice(list(ad_smooted_bigram[prev_word].keys()), 1, p=list(ad_smooted_bigram[prev_word].values()))[0])


    print("unigram_documents: ", [" ".join(word) for word in unigram_documents])
    print("li_documents: ", [" ".join(word) for word in li_documents])
    print("ad_documents: ", [" ".join(word) for word in ad_documents])


unigram_documents:  [['as'], ['we'], ['i'], ['off'], ['crab'], ['it'], ['oyster'], ['extra'], ['waitress'], ['tasti']]
li_documents:  [['hand'], ['i'], ['of'], ['it'], ['the'], ['a'], ['parker'], ['first'], ['is'], ['and']]
ad_documents:  [['seen'], ['went'], ['for'], ['big'], ['hitachino'], ['near'], ['recommend'], ['order'], ['want'], ['day']]
unigram_documents:  ['as dont', 'we is', 'i view', 'off in', 'crab urchin', 'it spice', 'oyster they', 'extra grill', 'waitress for', 'tasti achatz']
li_documents:  ['hand lobster', 'i is', 'of is', 'it by', 'the servic', 'a the', 'parker wa', 'first off', 'is as', 'and for']
ad_documents:  ['seen ill', 'went with', 'for a', 'big sandwich', 'hitachino shogun', 'near the', 'recommend thi', 'order at', 'want to', 'day but']
unigram_documents:  ['as dont but', 'we is way', 'i view not', 'off in avail', 'crab urchin chicken', 'it spice to', 'oyster they servic', 'extra grill as', 'waitress for about', 'tasti achatz unmatchedth']
li_documents:  ['ha

In [62]:
print("Unigram Documents:")
for index, document in enumerate(unigram_documents):
    prob = unigram_prob[word]
    for i in range(1, 20):
        prob *= unigram_prob[document[i]]
    print(f"{str(index+1).zfill(2)}: " + ' '.join(document))
    print(f"probability: {prob}")

print("\nlinear Interpolation Documents:")
for index, document in enumerate(li_documents):
    prob = unigram_prob[word]
    for i in range(1, 20):
        prob *= li_smooted_bigram[document[i-1]][document[i]]
    print(f"{str(index+1).zfill(2)}: " + ' '.join(document))
    print(f"probability: {prob}")

print("\nAbsolute Discount Documents:")
for index, document in enumerate(ad_documents):
    prob = unigram_prob[word]
    for i in range(1, 20):
        prob *= ad_smooted_bigram[document[i-1]][document[i]]
    print(f"{str(index+1).zfill(2)}: " + ' '.join(document))
    print(f"probability: {prob}")

Unigram Documents:
01: 'as dont but block from go the good thick well definit farm to heart so am veri meal would to
probability: 2.1805299954802233e-59
02: 'we is way the a fact in though foam in loud we it the # food had the okay fairli
probability: 7.408406314418808e-55
03: 'i view not more to is wa a and you yeah a flan it put thi sex minut to at
probability: 8.095873968537276e-56
04: 'off in avail jacketeven the tequila pork limit we ideal a at and to bona and friend wari it end
probability: 8.460164102534893e-68
05: 'crab urchin chicken to upon with door de hous deal clear fri husband view to back that i tlacoya more
probability: 4.1389872142428336e-69
06: 'it spice to to the finish and alreadi veri a i them fusili starsim to there howev price a us
probability: 6.768186099077475e-60
07: 'oyster they servic thought thi us post a sizabl crust we and id the chines lula next and or the
probability: 1.0431249647962343e-61
08: 'extra grill as scrambl # there your is benedict best bar a

# Reading Assignment — Belief or Bias in Information Retrieval (10pts)
In our class, we have learned both classical and modern information retrieval evaluation methods. And their shared goal is to assess if a retrieval system can satisfy users' information need. Such an evaluation directly leads to the subsequent optimization of retrieval system, e.g., optimize the ranking for click-through rates. But should a system please its users so as to improve the metrics or should it educate the users about what is right and wrong?

Let's read the paper ["Beliefs and biases in web search"](https://dl.acm.org/doi/10.1145/2484028.2484053), which is the best paper in SIGIR'2013. Based on the findings of this paper and current public concern/debate of the wide spread of misinformation on the web, what kind of suggestion do you want to give to Google and Bing to improve the situation? You can focus on the search evaluation, document retrieval and ranking, or any aspect related to the retrieval process.

# Extra Credits (5pts)

You are encouraged to further investigate the relation between classic language model and the trending Large Language Models. How LLMs differ from unigram and bigram models we implemented? It is okay to consult LLMs for this question :\) 

# Submission

This assignment has in total 100 points. The deadline is Feb 20 23:59 PDT. You should submit your report in **PDF** using the homework latex template, and submit your code (notebook)。